# DREAM AI — train our native model from random initialization

This notebook trains DREAM's own PyTorch transformer. It does **not** download or call another model, use a third-party AI API, or import pretrained weights. The checkpoint starts with random weights. The notebook defaults to DREAM's own byte-fallback BPE tokenizer for multilingual experiments; the legacy byte tokenizer remains selectable.

**Important:** the current architecture is an early research-scale model, not a capable general assistant. Training requires a substantial, high-quality dataset that you have permission to use. Do not mark rights/privacy/quality review as complete unless a person has actually completed those checks. Free GPU availability and session duration are not guaranteed.

In [ ]:
sft_completed = False
from pathlib import Path
import os, subprocess, sys, torch, json

REPO = Path('/content/dream-ai')
if not REPO.exists():
    subprocess.run(['git', 'clone', 'https://github.com/mjnr0477/dream-ai.git', str(REPO)], check=True)
os.chdir(REPO)

# Set True to keep checkpoints in Google Drive across Colab disconnects.
# Drive will ask you to authorize access. False keeps files only in this session.
USE_GOOGLE_DRIVE = False
if USE_GOOGLE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    CHECKPOINT_DIR = Path('/content/drive/MyDrive/DREAM AI/checkpoints')
else:
    CHECKPOINT_DIR = Path('checkpoints')
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
PRETRAIN_CHECKPOINT = CHECKPOINT_DIR / 'dream-bpe.pt'
SFT_CHECKPOINT = CHECKPOINT_DIR / 'dream-bpe-instruct.pt'

print('PyTorch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
else:
    print('No GPU detected. You can run a tiny smoke test on CPU, but substantial training will be slow.')

## 1. Upload the reviewed DREAM corpus

Prepare a JSONL file named `approved-records.jsonl`. Each line must contain `id`, `source`, `source_group`, `source_url` (HTTPS), `license`, `license_url` (HTTPS), `rights_basis`, `source_date` (`YYYY-MM-DD`), `rights_confirmed`, `privacy_reviewed`, `quality_reviewed`, `language`, and `text`. The three review fields must be `true` only after real human review. Use at least three independent source groups. Keep personal or confidential information out of the corpus. The generated manifest summarizes language and license counts and the date range for an audit before training. The notebook also runs a descriptive language-balance audit; its warnings require human review and it never approves the dataset.

In [ ]:
from google.colab import files
uploaded = files.upload()
if 'approved-records.jsonl' not in uploaded:
    raise RuntimeError('Please upload a file named approved-records.jsonl')
Path('data').mkdir(exist_ok=True)
Path('data/approved-records.jsonl').write_bytes(uploaded['approved-records.jsonl'])
print('Uploaded bytes:', Path('data/approved-records.jsonl').stat().st_size)

In [ ]:
Path('eval-results/corpus-audit.json').unlink(missing_ok=True)
corpus_audit_run = subprocess.run([
    sys.executable, 'model/audit_corpus.py',
    '--input', 'data/approved-records.jsonl',
    '--out', 'eval-results/corpus-audit.json'
], capture_output=True, text=True, check=True)
corpus_audit = json.loads(Path('eval-results/corpus-audit.json').read_text())
print('Privacy and provenance preflight (not approval):')
print(json.dumps(corpus_audit, indent=2))
if corpus_audit['error_count']:
    raise RuntimeError('Corpus provenance audit has blocking errors; inspect the report and source records.')
if corpus_audit['possible_pii_findings']:
    raise RuntimeError('Potential personal data was detected; inspect and redact the source records before training.')
if corpus_audit['unreviewed_records']:
    raise RuntimeError('Every record must pass actual human rights, privacy and quality review before training.')
subprocess.run([sys.executable, 'model/prepare_corpus.py', '--input', 'data/approved-records.jsonl', '--output', 'data/prepared'], check=True)
import json
manifest = json.loads(Path('data/prepared/manifest.json').read_text())
print(json.dumps(manifest, indent=2))
subprocess.run([
    sys.executable, 'model/audit_language_balance.py',
    '--input', 'data/approved-records.jsonl',
    '--out', 'eval-results/language-balance.json',
    '--min-records', '1000', '--max-share', '0.5'
], check=True)
language_balance = json.loads(Path('eval-results/language-balance.json').read_text())
print('Language balance review (not an approval):')
print(json.dumps(language_balance, indent=2))
for split in ('train', 'validation', 'test'):
    size = Path(f'data/prepared/{split}.txt').stat().st_size
    print(split, 'UTF-8 bytes:', size)
    if size < 258:
        raise RuntimeError(f'{split} split is too small for the default 256-byte context. Add more reviewed data before training.')

## 2. Train DREAM from scratch

The starter settings use a small micro-batch plus gradient accumulation to reduce peak memory. Increase steps only if the runtime can finish. Checkpoints are saved atomically. If you enable Google Drive above, they survive Colab session resets; otherwise they exist only until this runtime ends. Re-running this cell resumes from the existing DREAM checkpoint instead of restarting from random weights.

In [ ]:
# Use DREAM's byte-fallback BPE tokenizer for multilingual training.
# Set to "byte" to reproduce legacy byte-tokenizer runs.
TOKENIZER = "bpe"
BPE_VOCAB_SIZE = 1024

TRAIN_STEPS = 2000
MICRO_BATCH = 4
ACCUMULATION = 8
train_command = [
    sys.executable, 'model/train.py',
    '--data', 'data/prepared/train.txt',
    '--validation-data', 'data/prepared/validation.txt',
    '--out', str(PRETRAIN_CHECKPOINT),
    '--tokenizer', TOKENIZER,
    '--vocab-size', str(BPE_VOCAB_SIZE),
    '--steps', str(TRAIN_STEPS),
    '--batch-size', str(MICRO_BATCH),
    '--grad-accum-steps', str(ACCUMULATION),
    '--block-size', '256', '--d-model', '256', '--layers', '6', '--heads', '8',
    '--eval-every', '100', '--checkpoint-every', '250', '--eval-batches', '10'
]
if PRETRAIN_CHECKPOINT.exists():
    train_command += ['--resume', str(PRETRAIN_CHECKPOINT)]
    print('Resuming existing DREAM pretraining checkpoint with its saved tokenizer settings.')
MODEL_VOCAB_SIZE = BPE_VOCAB_SIZE if TOKENIZER == "bpe" else 256
subprocess.run([
    sys.executable, 'model/estimate_model.py',
    '--vocab-size', str(MODEL_VOCAB_SIZE),
    '--block-size', '256', '--d-model', '256',
    '--heads', '8', '--layers', '6',
    '--out', 'eval-results/model-budget.json'
], check=True)
budget = json.loads(Path('eval-results/model-budget.json').read_text())
print('Tokenizer:', TOKENIZER, 'model vocabulary size:', MODEL_VOCAB_SIZE)
print('Parameter count:', budget['parameter_count'])
print('FP32 Adam training-memory floor (bytes):',
      budget['parameter_memory']['fp32_adam_training_floor_bytes'])
print('Important:', budget['warning'])
subprocess.run(train_command, check=True)

## 3. Evaluate on the held-out test split

This reports next-token cross-entropy per token and per represented UTF-8 byte and a sample generation. These metrics do not prove reasoning, factual accuracy, safety or instruction-following ability.

In [ ]:
subprocess.run([
    sys.executable, 'model/evaluate.py',
    '--checkpoint', str(PRETRAIN_CHECKPOINT),
    '--data', 'data/prepared/test.txt',
    '--batches', '50', '--batch-size', '8',
    '--out', 'eval-results/dream-bpe.json'
], check=True)
report = json.loads(Path('eval-results/dream-bpe.json').read_text())
print('Bits per byte:', report['bits_per_byte'])
print('Perplexity per byte:', report['perplexity_per_byte'])
print('Sample:', report['sample_generation'])

## 4. Optional: instruction-tune DREAM on reviewed conversations

This second stage starts from the DREAM checkpoint you just trained. It does not load another model. Provide separate `sft-train.jsonl` and `sft-validation.jsonl` files. Every record must include HTTPS `source_url` and `license_url`, a valid `rights_basis`, strict `source_date`, and human-completed rights/privacy/quality review. The script masks user/system prompts and learns only from assistant responses. Their source groups must be disjoint from each other and from the held-out pretraining test sources in `data/prepared/manifest.json`.

In [ ]:
uploaded_sft = files.upload()
required_sft = ('sft-train.jsonl', 'sft-validation.jsonl')
sft_available = all(name in uploaded_sft for name in required_sft)
if sft_available:
    Path('data').mkdir(exist_ok=True)
    for name in required_sft:
        Path('data', name).write_bytes(uploaded_sft[name])
    print('Training bytes:', Path('data/sft-train.jsonl').stat().st_size)
    print('Validation bytes:', Path('data/sft-validation.jsonl').stat().st_size)
else:
    print('No complete SFT dataset uploaded. Skipping instruction tuning; pretraining remains available.')

In [ ]:
if sft_available:
    for split_name in ('train', 'validation'):
        audit_path = f'eval-results/sft-{split_name}-corpus-audit.json'
        Path(audit_path).unlink(missing_ok=True)
        subprocess.run([
            sys.executable, 'model/audit_corpus.py',
            '--input', f'data/sft-{split_name}.jsonl',
            '--out', audit_path
        ], check=True, capture_output=True, text=True)
        sft_audit = json.loads(Path(audit_path).read_text())
        print(f'SFT {split_name} privacy/provenance audit (not approval):')
        print(json.dumps(sft_audit, indent=2))
        if (sft_audit['error_count'] or sft_audit['possible_pii_findings']
                or sft_audit['unreviewed_records']):
            raise RuntimeError(
                f'SFT {split_name} data requires provenance, privacy or human-review corrections.'
            )
    SFT_STEPS = 1000
    sft_command = [
        sys.executable, 'model/sft.py',
        '--checkpoint', str(PRETRAIN_CHECKPOINT),
        '--train-data', 'data/sft-train.jsonl',
        '--validation-data', 'data/sft-validation.jsonl',
        '--pretraining-manifest', 'data/prepared/manifest.json',
        '--pretraining-test-data', 'data/prepared/test.txt',
        '--out', str(SFT_CHECKPOINT),
        '--steps', str(SFT_STEPS),
        '--batch-size', '2', '--grad-accum-steps', '4',
        '--eval-every', '100', '--checkpoint-every', '250',
        '--eval-batches', '10'
    ]
    if SFT_CHECKPOINT.exists():
        sft_command += ['--resume', str(SFT_CHECKPOINT)]
        print('Resuming existing DREAM instruction-tuning checkpoint.')
    subprocess.run(sft_command, check=True)
    sft_completed = True
else:
    print('Instruction tuning skipped. The native pretraining checkpoint is still available.')

In [ ]:
if sft_completed:
    subprocess.run([
        sys.executable, 'model/evaluate.py',
        '--checkpoint', str(SFT_CHECKPOINT),
        '--data', 'data/prepared/test.txt',
        '--batches', '50', '--batch-size', '8',
        '--out', 'eval-results/dream-bpe-instruct.json'
    ], check=True)
    sft_report = json.loads(Path('eval-results/dream-bpe-instruct.json').read_text())
    print('Instruction checkpoint bits per byte:', sft_report['bits_per_byte'])
    print('Sample:', sft_report['sample_generation'])
else:
    print('No instruction checkpoint to evaluate; pretraining evaluation above is complete.')

## 6. Package the reproducible DREAM training artifacts

The bundle contains the checkpoints, evaluation reports, exact pretraining train/validation/test text splits and their manifest. If SFT completed, it also includes the reviewed SFT train/validation JSONL files and instruction checkpoint. Keep this bundle private if your reviewed data should not be shared.

In [ ]:
from zipfile import ZipFile, ZIP_DEFLATED
from hashlib import sha256
artifact_entries = [
    (PRETRAIN_CHECKPOINT, 'checkpoints/dream-bpe.pt'),
    (Path('eval-results/dream-bpe.json'), 'eval-results/dream-bpe.json'),
    (Path('eval-results/language-balance.json'), 'eval-results/language-balance.json'),
    (Path('eval-results/corpus-audit.json'), 'eval-results/corpus-audit.json'),
    (Path('eval-results/model-budget.json'), 'eval-results/model-budget.json'),
    (Path('data/prepared/manifest.json'), 'data/prepared/manifest.json'),
    (Path('data/prepared/train.txt'), 'data/prepared/train.txt'),
    (Path('data/prepared/validation.txt'), 'data/prepared/validation.txt'),
    (Path('data/prepared/test.txt'), 'data/prepared/test.txt')
]
if sft_completed:
    artifact_entries += [
        (SFT_CHECKPOINT, 'checkpoints/dream-bpe-instruct.pt'),
        (Path('eval-results/dream-bpe-instruct.json'), 'eval-results/dream-bpe-instruct.json'),
        (Path('data/sft-train.jsonl'), 'data/sft-train.jsonl'),
        (Path('data/sft-validation.jsonl'), 'data/sft-validation.jsonl'),
        (Path('eval-results/sft-train-corpus-audit.json'), 'eval-results/sft-train-corpus-audit.json'),
        (Path('eval-results/sft-validation-corpus-audit.json'), 'eval-results/sft-validation-corpus-audit.json')
    ]
bundle = Path('dream-training-artifacts.zip')
with ZipFile(bundle, 'w', compression=ZIP_DEFLATED) as archive:
    for path, archive_name in artifact_entries:
        if not path.is_file():
            raise FileNotFoundError(f'Missing required training artifact: {path}')
        archive.write(path, arcname=archive_name)
    checksum_lines = []
    for path, archive_name in artifact_entries:
        digest = sha256(path.read_bytes()).hexdigest()
        checksum_lines.append(f'{digest}  {archive_name}')
    archive.writestr('SHA256SUMS.txt', '\n'.join(checksum_lines) + '\n')
print('Bundle size (bytes):', bundle.stat().st_size)
files.download(str(bundle))

## 7. Connect the trained native DREAM model to the live website (temporary session)

Run this only after instruction tuning completed successfully. It starts the **native DREAM checkpoint** in this Colab session and creates a temporary HTTPS Cloudflare Quick Tunnel so the Render website can reach it. It does not use another AI provider. The tunnel and model stop when Colab disconnects or the session ends; this is for the first live connection test, not permanent hosting.

After this cell prints the three environment variables, add them to the existing `dream-ai-live` Render service under **Environment**: `DREAM_MODEL_URL`, `DREAM_MODEL_NAME`, and `DREAM_MODEL_TOKEN`. Save changes and wait for the web service to redeploy. Keep the token private. Never paste it into GitHub, source files, or public chats.

In [ ]:
# A trained assistant checkpoint is required; pretraining-only weights are not enough.
if not sft_completed or not SFT_CHECKPOINT.is_file():
    raise RuntimeError(
        "Instruction tuning has not completed. Upload separately reviewed SFT train/validation "
        "data, complete the SFT and held-out evaluation cells, then run this connection cell."
    )

if not Path('eval-results/dream-bpe-instruct.json').is_file():
    raise RuntimeError('Run the held-out instruction-checkpoint evaluation cell before connecting the website.')

import re, secrets, time, urllib.request

RUNTIME_MODEL_NAME = "dream-native-v1"
RUNTIME_TOKEN = secrets.token_urlsafe(32)
runtime_env = os.environ.copy()
runtime_env.update({
    "DREAM_RUNTIME_HOST": "127.0.0.1",
    "DREAM_RUNTIME_PORT": "8081",
    "DREAM_RUNTIME_TOKEN": RUNTIME_TOKEN,
    "DREAM_MODEL_NAME": RUNTIME_MODEL_NAME,
    "DREAM_CPU_THREADS": "2",
})

RUNTIME_PROCESS = subprocess.Popen(
    [
        sys.executable, "model/serve.py",
        "--checkpoint", str(SFT_CHECKPOINT),
        "--host", "127.0.0.1",
        "--port", "8081",
        "--model-name", RUNTIME_MODEL_NAME,
        "--cpu-threads", "2",
    ],
    cwd=str(REPO),
    env=runtime_env,
    stdout=open('/content/dream-native-runtime.log', 'w'),
    stderr=subprocess.STDOUT,
)

runtime_health = None
for _ in range(90):
    if RUNTIME_PROCESS.poll() is not None:
        log_path = Path('/content/dream-native-runtime.log')
        details = log_path.read_text(encoding='utf-8', errors='replace')[-3000:] if log_path.exists() else 'No runtime log was created.'
        raise RuntimeError('Native DREAM runtime exited during startup:\\n' + details)
    try:
        with urllib.request.urlopen("http://127.0.0.1:8081/health", timeout=3) as response:
            runtime_health = json.loads(response.read().decode("utf-8"))
        break
    except Exception:
        time.sleep(1)

if not runtime_health or runtime_health.get("engine") != "dream-native-transformer":
    raise RuntimeError("Native DREAM runtime did not pass its local identity check.")
if not runtime_health.get("completed_training_steps") or not re.fullmatch(
    r"[a-f0-9]{64}", runtime_health.get("checkpoint_sha256", "")
):
    raise RuntimeError("Runtime checkpoint identity or training-step verification failed.")

CLOUDFLARED_PATH = Path("/content/cloudflared")
if not CLOUDFLARED_PATH.exists():
    urllib.request.urlretrieve(
        "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64",
        str(CLOUDFLARED_PATH),
    )
    os.chmod(CLOUDFLARED_PATH, 0o755)

CLOUDFLARED_LOG = Path("/content/dream-cloudflared.log")
with CLOUDFLARED_LOG.open("w") as log_handle:
    TUNNEL_PROCESS = subprocess.Popen(
        [str(CLOUDFLARED_PATH), "tunnel", "--no-autoupdate", "--url", "http://127.0.0.1:8081"],
        stdout=log_handle,
        stderr=subprocess.STDOUT,
    )

TUNNEL_URL = None
for _ in range(90):
    if TUNNEL_PROCESS.poll() is not None:
        raise RuntimeError("The temporary HTTPS tunnel exited before it became ready.")
    tunnel_log = CLOUDFLARED_LOG.read_text(encoding="utf-8", errors="replace")
    match = re.search(r"https://[a-z0-9-]+\\.trycloudflare\\.com", tunnel_log)
    if match:
        TUNNEL_URL = match.group(0)
        break
    time.sleep(1)

if not TUNNEL_URL:
    raise RuntimeError("No temporary HTTPS URL appeared. Check the Cloudflare tunnel log.")

with urllib.request.urlopen(TUNNEL_URL + "/health", timeout=20) as response:
    public_health = json.loads(response.read().decode("utf-8"))
if public_health.get("checkpoint_sha256") != runtime_health.get("checkpoint_sha256"):
    raise RuntimeError("Public tunnel health identity does not match the local DREAM checkpoint.")

print("Native DREAM checkpoint verified:", runtime_health["checkpoint_sha256"])
print("Training steps:", runtime_health["completed_training_steps"])
print("Device:", runtime_health["device"])
print()
print("Copy these values into the existing Render service environment:")
print("DREAM_MODEL_URL=" + TUNNEL_URL + "/v1")
print("DREAM_MODEL_NAME=" + RUNTIME_MODEL_NAME)
print("DREAM_MODEL_TOKEN=" + RUNTIME_TOKEN)
print()
print("Keep this Colab session running while testing the website. The tunnel is temporary.")
